### Set Up

In [2]:
import os
os.environ["KERAS_BACKEND"] = "torch"

In [3]:
import numpy as np
import keras
import torch
import tensorflow as tf
import re


In [4]:
# Quick health check to prove it works:
print("Is PyTorch utilizing the GPU?:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))

Is PyTorch utilizing the GPU?: True
Device Name: NVIDIA GeForce RTX 5050 Laptop GPU


### Load dataset

In [5]:
# Downloading an abbreviated collection of Shakespeare’s work
filename = r"C:\Users\PRASHANTH N\PycharmProjects\MTechSem2\DLRL\BOOKS\1\Experiments\ChatText\chat.txt"
chat = open(filename, "r", encoding="utf-8").read()
print(chat[:250]), len(chat)

5/31/26, 12:01 PM - +91 93802 39584: Good afternoon 🍀🥺
6/4/26, 12:40 PM - Prashanth: I miss u ❤️
6/4/26, 8:34 PM - +91 93802 39584: U didn't reply me back , u like silence games
6/4/26, 8:34 PM - +91 93802 39584: Now ur telling this
6/4/26, 8:46 PM -


(None, 217503)

### Data preprocess

In [6]:
# Replace Name
chat = chat.replace("+91 93802 39584", "Srishti")
print(chat[:250]), len(chat)

5/31/26, 12:01 PM - Srishti: Good afternoon 🍀🥺
6/4/26, 12:40 PM - Prashanth: I miss u ❤️
6/4/26, 8:34 PM - Srishti: U didn't reply me back , u like silence games
6/4/26, 8:34 PM - Srishti: Now ur telling this
6/4/26, 8:46 PM - Prashanth: Comon. Make 


(None, 202167)

In [7]:
# Remove Time

timestamp_pattern = r"\d{1,2}/\d{1,2}/\d{2},\s\d{1,2}:\d{2}\s(?:AM|PM)\s-\s"
chat = re.sub(timestamp_pattern, "", chat)
print(chat[:250]), len(chat)

Srishti: Good afternoon 🍀🥺
Prashanth: I miss u ❤️
Srishti: U didn't reply me back , u like silence games
Srishti: Now ur telling this
Prashanth: Comon. Make it easy for me talk
Prashanth: I did dint wanted to text u even tho i wanted
Prashanth: We fo


(None, 139027)

In [8]:
# Splitting text into chunks for language model training
sequence_length = 100
def split_input(input, sequence_length):
    for i in range(0, len(input), sequence_length):
        yield input[i : i + sequence_length]
features = list(split_input(chat[:-1], sequence_length))
labels = list(split_input(chat[1:], sequence_length))
dataset = tf.data.Dataset.from_tensor_slices((features, labels))

In [9]:
features[:1], features[-1:]

(["Srishti: Good afternoon 🍀🥺\nPrashanth: I miss u ❤️\nSrishti: U didn't reply me back , u like silence g"],
 ['Right. U are my family now'])

In [10]:
labels[:1], labels[-1:]

(["rishti: Good afternoon 🍀🥺\nPrashanth: I miss u ❤️\nSrishti: U didn't reply me back , u like silence ga"],
 ['ight. U are my family now\n'])

In [11]:
x, y = next(dataset.as_numpy_iterator())
x, y

(b"Srishti: Good afternoon \xf0\x9f\x8d\x80\xf0\x9f\xa5\xba\nPrashanth: I miss u \xe2\x9d\xa4\xef\xb8\x8f\nSrishti: U didn't reply me back , u like silence g",
 b"rishti: Good afternoon \xf0\x9f\x8d\x80\xf0\x9f\xa5\xba\nPrashanth: I miss u \xe2\x9d\xa4\xef\xb8\x8f\nSrishti: U didn't reply me back , u like silence ga")

In [12]:
# Learning a character-level vocabulary with the TextVectorization layer
tokenizer = keras.layers.TextVectorization(
    standardize=None,
    split="character",
    output_sequence_length=sequence_length,
)
tokenizer.adapt(dataset.map(lambda text, labels: text))

In [13]:
vocabulary_size = tokenizer.vocabulary_size()
vocabulary_size

173

In [14]:
dataset = dataset.map(lambda features, labels: (tokenizer(features), tokenizer(labels)), num_parallel_calls=8,)
training_data = dataset.shuffle(10_000).batch(64).cache()

### Model

In [15]:
# Building a miniature language model
embedding_dim = 256
hidden_dim = 1024
inputs = keras.layers.Input(shape=(sequence_length,), dtype="int", name="token_ids")
x = keras.layers.Embedding(vocabulary_size, embedding_dim)(inputs)
x = keras.layers.GRU(hidden_dim, return_sequences=True)(x)
x = keras.layers.Dropout(0.3)(x)
outputs = keras.layers.Dense(vocabulary_size, activation="softmax")(x)
model = keras.Model(inputs, outputs)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ token_ids (InputLayer)          │ (None, 100)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 100, 256)       │        44,288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 100, 1024)      │     3,938,304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 100, 1024)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 100, 173)       │       177,325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,159,917 (15.87 MB)

 Trainable params: 4,159,917 (15.87 MB)

 Non-trainable params: 0 (0.00 B)

### Training

In [16]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor="loss",
    restore_best_weights=True,
    patience=2,
)

In [17]:
# Training a miniature language mode
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["sparse_categorical_accuracy"],
)
model.fit(training_data, epochs=30, callbacks=[early_stopping])

Epoch 1/30
22/22 ━━━━━━━━━━━━━━━━━━━━ 24s 905ms/step - loss: 4.0241 - sparse_categorical_accuracy: 0.1473
Epoch 2/30
22/22 ━━━━━━━━━━━━━━━━━━━━ 20s 920ms/step - loss: 2.6653 - sparse_categorical_accuracy: 0.3113
Epoch 3/30
22/22 ━━━━━━━━━━━━━━━━━━━━ 21s 931ms/step - loss: 2.2199 - sparse_categorical_accuracy: 0.4040
Epoch 4/30
22/22 ━━━━━━━━━━━━━━━━━━━━ 20s 909ms/step - loss: 2.0202 - sparse_categorical_accuracy: 0.4389
Epoch 5/30
22/22 ━━━━━━━━━━━━━━━━━━━━ 20s 910ms/step - loss: 1.9090 - sparse_categorical_accuracy: 0.4660
Epoch 6/30
22/22 ━━━━━━━━━━━━━━━━━━━━ 20s 908ms/step - loss: 1.8257 - sparse_categorical_accuracy: 0.4878
Epoch 7/30
22/22 ━━━━━━━━━━━━━━━━━━━━ 20s 903ms/step - loss: 1.7595 - sparse_categorical_accuracy: 0.5019
Epoch 8/30
22/22 ━━━━━━━━━━━━━━━━━━━━ 20s 918ms/step - loss: 1.7012 - sparse_categorical_accuracy: 0.5156
Epoch 9/30
22/22 ━━━━━━━━━━━━━━━━━━━━ 20s 914ms/step - loss: 1.6459 - sparse_categorical_accuracy: 0.5306
Epoch 10/30
22/22 ━━━━━━━━━━━━━━━━━━━━ 20s 905

### Inference

In [18]:
# Modifying the language model for autoregressive inference
inputs = keras.Input(shape=(1,), dtype="int", name="token_ids")
input_state = keras.Input(shape=(hidden_dim,), name="state")
x = keras.layers.Embedding(vocabulary_size, embedding_dim)(inputs)
x, output_state = keras.layers.GRU(hidden_dim, return_state=True)(x, initial_state=input_state)
outputs = keras.layers.Dense(vocabulary_size, activation="softmax")(x)
generation_model = keras.Model(inputs=(inputs, input_state), outputs=(outputs, output_state), )
generation_model.set_weights(model.get_weights())
generation_model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ token_ids           │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 1, 256)    │     44,288 │ token_ids[0][0]   │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ state (InputLayer)  │ (None, 1024)      │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru_1 (GRU)         │ [(None, 1024),    │  3,938,304 │ embedding_1[0][0… │
│                     │ (None, 1024)]     │            │ state[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 173)       │    177,325 │ gru_1[0][0]       │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 4,159,917 (15.87 MB)

 Trainable params: 4,159,917 (15.87 MB)

 Non-trainable params: 0 (0.00 B)

In [41]:
tokens = tokenizer.get_vocabulary()
token_ids = range(vocabulary_size)
char_to_id = dict(zip(tokens, token_ids))
id_to_char = dict(zip(token_ids, tokens))
prompt = """Prashanth: Hi. What u doing?
Srishti: Missing you."""

In [42]:
# Using a fixed prompt to compute a language model’s starting state
input_ids = [char_to_id[c] for c in prompt]
state = keras.ops.zeros(shape=(1, hidden_dim))
for token_id in input_ids:
    inputs = keras.ops.expand_dims([token_id], axis=0)
    predictions, state = generation_model.predict((inputs, state), verbose=0)
    print(predictions, state)

[[5.22127311e-06 5.80535198e-06 1.90616320e-05 1.16041410e-04
  6.96023740e-03 3.70382695e-05 3.83575883e-04 1.04080013e-03
  1.64455257e-03 9.51346278e-01 1.36717921e-04 1.05337310e-03
  1.64559489e-04 1.20311626e-03 2.09197060e-06 3.39046237e-04
  1.17956568e-03 1.86429636e-04 7.21380056e-04 9.29302914e-05
  4.05905710e-04 7.20485186e-05 2.49828940e-04 2.21052370e-03
  7.18256924e-03 5.11296712e-05 1.43073594e-05 1.44756377e-05
  4.58185386e-04 1.35078735e-04 3.21118860e-04 3.07238079e-04
  4.55408212e-04 2.91445001e-04 1.05374740e-04 5.32591148e-05
  6.95669441e-04 9.09340975e-04 6.98725227e-03 1.93793283e-04
  4.92566032e-04 1.98662543e-04 1.31464185e-06 1.79603958e-05
  9.07203648e-05 1.85268538e-04 1.85565965e-04 6.57492928e-05
  1.80540839e-04 3.61124694e-05 5.15882217e-04 2.18702276e-04
  4.03491606e-04 2.42717633e-05 3.46690940e-05 5.40297151e-05
  4.29405458e-03 3.09708548e-06 2.85413233e-04 2.15300519e-04
  4.67090795e-05 5.36807011e-06 1.46751723e-03 1.38888790e-04
  9.5132

In [43]:
# Predicting with the language model a token at a time
generated_ids = []
max_length = 250
for i in range(max_length):
    next_char = int(np.argmax(predictions, axis=-1)[0])
    generated_ids.append(next_char)
    inputs = keras.ops.expand_dims([next_char], axis=0)
    predictions, state = generation_model.predict((inputs, state), verbose=0)

In [44]:
output = "".join([id_to_char[token_id] for token_id in generated_ids])
print(prompt + output)

Prashanth: Hi. What u doing?
Srishti: Missing you. I will be sometimes I feel hot to start then
Srishti: We can talk tomorrow
Srishti: What are u doing ?
Prashanth: U slept?
Srishti: Why u doing ?
Prashanth: U slept?
Srishti: Why u doing ?
Prashanth: U slept?
Srishti: Why u doing ?
Prashanth: U slep
